[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.1 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [28]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # pass  # extract patches, apply kernel, handle stride/padding
    # C_out = number of filters of shape kH and kW in total.
    if padding > 0:
      x = F.pad(x, [padding]*4) #-> for 4 sides
    B, C_in, H, W = x.shape
    C_out, _, kH, kW = weight.shape
    W_out = (W-kW) // stride + 1
    H_out = (H-kH) // stride + 1
    x_unfold = x.unfold(2, kH, stride).unfold(3, kW, stride)
    # (B, C_in, H_out, W_out, kH, kW)
    # w: (C_out, C_in, kH, kW)
    # output: (B, C_out, H_out, W_out)
    output = torch.einsum('bihwkj, oikj -> bohw', x_unfold, weight)
    print(x.shape, weight.shape)
    if bias is not None:
      #bias: (C_out,) -> reshape to (1, C_out, 1, 1) to match the output shape
      print(bias.shape)
      output += bias.reshape(1, -1, 1, 1)
    return output




In [29]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

torch.Size([1, 3, 8, 8]) torch.Size([16, 3, 3, 3])
Output: torch.Size([1, 16, 6, 6])
torch.Size([1, 3, 8, 8]) torch.Size([16, 3, 3, 3])
Match: True


In [30]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
torch.Size([1, 3, 8, 8]) torch.Size([16, 3, 3, 3])
  ✅ [1/5] Output shape (1.7ms)
torch.Size([2, 3, 8, 8]) torch.Size([4, 3, 3, 3])
torch.Size([4])
  ✅ [2/5] Matches F.conv2d (3.3ms)
torch.Size([1, 1, 7, 7]) torch.Size([1, 1, 3, 3])
  ✅ [3/5] With padding (4.0ms)
torch.Size([1, 1, 8, 8]) torch.Size([1, 1, 3, 3])
  ✅ [4/5] With stride (3.0ms)
torch.Size([1, 1, 4, 4]) torch.Size([2, 1, 3, 3])
  ✅ [5/5] Gradient flow (1.5ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (13.4ms total)
  Progress saved. Run status() to see your dashboard.

